# Chess Move Error Detection with a Bidirectional Transformer

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gabrielmario-tavini/chess-move-error-detection/blob/main/chess_error_detection.ipynb)

**Gabrielmario Tavini** - [GitHub](https://github.com/gabrielmario-tavini) - [LinkedIn](https://www.linkedin.com/in/gabrielmario-tavini/)

**Result: Accuracy@1 = 0.96894** on the 50,000-game public test set, with
**5,089,705** trainable parameters under a 6,000,000 budget. Non-neural baseline: 0.426.

Final project of the Deep Learning course, M.Sc. in Automation Engineering,
University of Bologna (A.Y. 2025/26), graded 30/30.

## Problem

Each example is the first **40 plies** of a real chess game in SAN notation.
Exactly one of the first **10 plies** has been replaced by the move played at the
same ply in a *different* real game. The task is to predict which of the 10
positions was corrupted: a 10-class classification problem evaluated on top-1
accuracy.

```text
original : e4 e5 Nf3 Nc6 Bb5 a6 ...
corrupted: e4 e5 Nf3 e6  Bb5 a6 ...     -> target = 4
```

## Constraints

- **No chess knowledge**: no rules, legality checks, engines, opening books or
  hand-built features. Moves are opaque tokens and everything is learned from data.
- **No externally pre-trained models.**
- The replaced move (`correct_move`) may be used as a training label but **never as
  a model input**.
- **At most 6,000,000 trainable parameters.**

## How to run

| Path | Steps | Time on a T4 |
|---|---|---|
| Reproduce the result from the trained weights | Data cell, Steps 0, 1.1, 1.2, 2, 2.1, 3, 3.1, 10 | ~5 min |
| Full training | Data cell, Steps 0, 1, 2, 3, 6, 7c, 8b | ~2.5 h |

Steps that document ablations and discarded experiments are disabled by flags in
Step 0; their recorded results are reported in the text after each of them.


## Data

The dataset (400,000 training games, 50,000 test games, columns `sequence`,
`error_position`, `correct_move`) was provided with the course and is included in
this repository under `data/`, gzip-compressed. pandas reads the compressed files
directly.

The cell below loads it, runs a few consistency checks and defines the official
metric, top-1 accuracy over positions 1 to 10.


In [ ]:
import os
import numpy as np
import pandas as pd

DATA_DIR   = "data"                                            # edit if needed
TRAIN_PATH = os.path.join(DATA_DIR, "chess_error_detection_train.csv.gz")
TEST_PATH  = os.path.join(DATA_DIR, "chess_error_detection_test.csv.gz")

# When the notebook is opened alone (e.g. "Open in Colab"), the data/ folder of
# the repository is not on disk: fetch the two files from GitHub.
REPO_RAW = "https://raw.githubusercontent.com/gabrielmario-tavini/chess-move-error-detection/main/"
os.makedirs(DATA_DIR, exist_ok=True)
for path in (TRAIN_PATH, TEST_PATH):
    if not os.path.exists(path):
        import urllib.request
        print("downloading", path)
        urllib.request.urlretrieve(REPO_RAW + path, path)

train_df = pd.read_csv(TRAIN_PATH)                             # (400000, 3)
test_df  = pd.read_csv(TEST_PATH)                              # ( 50000, 3)
print("train", train_df.shape, "| test", test_df.shape)

for df in (train_df, test_df):
    assert set(df.columns) == {"sequence", "error_position", "correct_move"}
    assert (df["sequence"].str.split().str.len() == 40).all()
    assert df["error_position"].between(1, 10).all()


def accuracy_at_1(y_true, probabilities):
    """Top-1 accuracy. y_true in 1..10, probabilities of shape (N, 10)."""
    y_true = np.asarray(y_true)                                # (N,)
    probabilities = np.asarray(probabilities)                  # (N,10)
    assert probabilities.ndim == 2 and probabilities.shape[1] == 10
    assert len(y_true) == len(probabilities)
    return np.mean(probabilities.argmax(1) + 1 == y_true)


---
---

# SOLUTION

**Final result: Accuracy@1 = 0.96894 on the public test set, 5,089,705 trainable parameters.**

**Index**

| Step | Content | Needed for the final result | Time on T4 |
|---|---|---|---|
| 0 | Setup and configuration | yes | 10 s |
| 1 | Signal analysis and non-neural baselines | no, but motivates the architecture | 5 min |
| 2 | Tokenization and data pipeline | yes | 2 min |
| 3 | Model architecture | yes | 10 s |
| 4 | Pipeline check (overfit on 512 examples) | no, diagnostic | 3 min |
| 5 | Experiments without pre-training (ablations) | no, used for comparison | 90 min |
| 6 | Self-supervised pre-training (MLM) | yes | 65 min |
| 7 | Fine-tuning | yes | 30 min |
| 7b | How much pre-training is needed | no, selects the snapshot | 150 min |
| 7c | Multi-task fine-tuning: pointer + MLM | yes | 40 min |
| 7d | The lambda trade-off (discarded) | no | 40 min |
| 8 | Evaluation on the test set | yes | 1 min |
| 8b | Pseudo-log-likelihood scoring | yes | 20 min |
| 8c | Post-hoc calibration (no gain) | no | 2 min |
| 9 | Discarded experiment: longer fine-tuning | no | 60 min |
| 10 | Loading the trained weights | yes (fast path) | 1 min |
| 11 | Summary of results | - | - |
| 12 | Exporting the weights | no, utility | - |

**Steps 4, 5, 7b, 7d, 8c and 9 are disabled** by boolean flags in Step 0, or skip
automatically when the variables they need are absent. They are kept because they
document the ablations and the discarded experiments that justify the final
choices, but they do not need to be re-run: their results are reported in the text
cells that follow each of them.

**Fast path, reproduces the final result without training:**
run the Data cell, then Steps 0, 1.1, 1.2, 2, 2.1, 3, 3.1 and 10. About 5 minutes.

**Full training path:**
Data cell, Steps 0, 1, 2, 3, 6, 7c, 8b. About two and a half hours.

**How the final model was reached**

| Change | Val | Test |
|---|---|---|
| non-neural baseline (Step 1.6) | - | 0.426 |
| transformer, no pre-training | 0.9324 | - |
| + MLM pre-training, 8 epochs | 0.9549 | 0.9565 |
| + MLM pre-training, 20 epochs | 0.9600 | - |
| + multi-task fine-tuning | 0.9594 | 0.9595 |
| **+ PLL scoring** | **0.9682** | **0.9689** |


**Platform.** The notebook runs on Colab, Kaggle and locally. Step 10 downloads
the trained weights from the GitHub Release of this repository when they are not
already on disk. Training times in the table above refer to a T4 GPU.

## STEP 0 - Setup and configuration

Every hyperparameter lives in this cell. Experiments are run by editing only
here and re-running the relevant steps, without creating new cells.

The values below are those of the final model. The flags are all `False`: the
notebook is in its reproduction configuration, where Step 10 loads the trained
weights and reproduces the result without training.

In [ ]:
import os, math, glob, shutil, numpy as np, pandas as pd, tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from collections import Counter

AUTOTUNE = tf.data.AUTOTUNE
keras.utils.set_random_seed(0)          # reproducibility
print("GPU:", tf.config.list_physical_devices('GPU'))

# ---------- execution flags ----------
RUN_SANITY_CHECK   = False   # Step 4:  overfit on 512 examples
RUN_ABLATIONS      = False   # Step 5:  training without pre-training
RUN_PRETRAIN       = False   # Step 6:  MLM pre-training
RUN_FINETUNE       = False   # Step 7:  single fine-tuning run
RUN_SNAPSHOT_SWEEP = False   # Step 7b: fine-tuning from each MLM snapshot
RUN_JOINT_FT       = False   # Step 7c: multi-task fine-tuning, final model
RUN_PLL            = False   # Step 8b: pseudo-log-likelihood scoring
RUN_LONG_FT        = False   # Step 9:  24-epoch fine-tuning (discarded)

# ---------- architecture ----------
D_MODEL   = 256     # width. Parameter count scales with d^2
N_HEADS   = 8       # key_dim = D_MODEL // N_HEADS, see Step 3
FF_DIM    = 1024    # 4 * D_MODEL
N_BLOCKS  = 4       # 4 -> 4.13M params, 5 -> 4.92M, 6 -> 5.71M
DROPOUT   = 0.1
MAXLEN    = 40

# ---------- MLM pre-training ----------
MLM_EPOCHS    = 20                  # 8 in the first version, see Step 7b
MLM_LR        = 5e-4
MASK_PROB     = 0.15
MLM_SNAPSHOTS = [8, 12, 16, 20]     # epochs at which encoder and head are saved
N_MLM_VAL     = 10000               # clean games held out from pre-training

# ---------- fine-tuning ----------
FT_EPOCHS    = 12      # see Step 9 before increasing: moves together with COS_ALPHA
FT_LR        = 1e-4
COS_ALPHA    = 0.05    # final learning rate = COS_ALPHA * FT_LR
LABEL_SMOOTH = 0.05
PATIENCE     = 4
USE_EMA      = False   # exponential moving average of the weights
EMA_MOMENTUM = 0.999
LAMBDA_MLM   = 0.3     # weight of the MLM loss in multi-task fine-tuning, see Step 7d

# ---------- inference ----------
GAMMA_GRID = [0.0, 0.5, 1.0, 1.5, 2.0, 3.0, 5.0, 10.0]   # weight of the PLL term
GAMMA_BEST = 2.0                                          # selected in Step 8b

# ---------- shared ----------
BATCH  = 256
N_VAL  = 20000
WD     = 1e-4

# Per-position C3 baseline (Step 1.6). Fallback if Step 1 is not executed.
BASELINE = np.array([0.000,0.002,0.303,0.400,0.553,0.680,0.701,0.645,0.546,0.421])

GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## STEP 1 - Signal analysis and non-neural baselines

Before choosing an architecture we want to know **where the signal is**. Three
sentences of the specification determine it.

**First sentence.** The replaced move is taken from the same ply of another real
game. The marginal distribution at ply *i* is therefore identical for the true
move and for the corrupted one, and any statistic of the form "this move is rare
at ply *i*" has zero expected signal. The signal can only live in the
**conditional** distribution.

**Second sentence.** Plies 11 to 40 are part of the input. They are the
continuation of the **original** game, hence consistent with the true move and
inconsistent with the corrupted one. This calls for an encoder that also looks
forward.

**Third sentence.** Exactly one of the first ten is corrupted. This is an
exclusivity constraint, so the decision is a competition among ten candidates.

Steps 1.1 to 1.6 measure how much signal each source carries.

### 1.1 Vocabulary and per-position entropy

The vocabulary size determines the cost of the embedding, which is the single
heaviest item in the parameter budget. The per-ply entropy says how predictable
each position is, and therefore how hard it will be to check.

In [ ]:
M = np.array(train_df["sequence"].str.split().tolist())      # (400000, 40)
V = len(set(M.ravel().tolist()))
print("total vocabulary V =", V)

for i in [0, 1, 2, 4, 9, 19, 39]:
    c = Counter(M[:, i])
    p = np.array(list(c.values()), dtype=float); p /= p.sum()
    H = -(p * np.log2(p)).sum()
    print(f"ply {i+1:2d}: distinct={len(c):5d}  H={H:5.2f} bits  most frequent={c.most_common(1)[0]}")

c_all = Counter(M.ravel().tolist())
freqs = np.array(sorted(c_all.values(), reverse=True), dtype=float)
cum = np.cumsum(freqs) / freqs.sum()
for t in [0.99, 0.999, 0.9999]:
    print(f"coverage {t}: {int(np.searchsorted(cum, t)) + 1} tokens")

total vocabulary V = 3740
ply  1: distinct=   20  H= 2.06 bits  most frequent=(np.str_('e4'), 219373)
ply  2: distinct=   20  H= 3.07 bits  most frequent=(np.str_('e5'), 101645)
ply  3: distinct=   95  H= 3.85 bits  most frequent=(np.str_('Nf3'), 109714)
ply  5: distinct=  282  H= 4.73 bits  most frequent=(np.str_('Nc3'), 63204)
ply 10: distinct=  612  H= 6.19 bits  most frequent=(np.str_('Nf6'), 33908)
ply 20: distinct= 1137  H= 7.43 bits  most frequent=(np.str_('O-O'), 25241)
ply 40: distinct= 2306  H= 8.94 bits  most frequent=(np.str_('f5'), 4590)
coverage 0.99: 920 tokens
coverage 0.999: 1614 tokens
coverage 0.9999: 2682 tokens


Result: **V = 3740**, entropy rising from 2.06 bits at ply 1 to 8.94 at ply 40.

The vocabulary is small, so there is no need to truncate it: `Embedding(3741, 256)`
costs just under one million parameters. The rising entropy does tell us that the
ten classes are not equally hard: 20 distinct moves occur at ply 1, 612 at ply 10.

### 1.2 Reconstructing the clean games and counting prefixes

On the training set we know `error_position` and `correct_move`, so the original
games can be reconstructed. This uses `correct_move` as a **training label**,
never as a model input, and therefore respects the constraint in the
specification.

The clean games serve two purposes: building the opening-prefix statistics used
by the baselines, and providing the data for the pre-training of Step 6.

In [ ]:
clean = M.copy()                                             # (400000, 40)
clean[np.arange(len(clean)), train_df["error_position"].to_numpy() - 1] = \
    train_df["correct_move"].to_numpy()
print("clean games:", clean.shape)

clean games: (400000, 40)


In [ ]:
# Prefix statistics. Only used by the baselines of Steps 1.3 to 1.6;
# the fast path of Step 10 does not need this cell.
prefix_count = [Counter() for _ in range(11)]
prefix_count[0][()] = len(clean)
for L in range(1, 11):
    prefix_count[L] = Counter(map(tuple, clean[:, :L].tolist()))
    print(f"distinct prefixes of length {L:2d}: {len(prefix_count[L]):7d}")

distinct prefixes of length  1:      20
distinct prefixes of length  2:     354
distinct prefixes of length  3:    3285
distinct prefixes of length  4:   14504
distinct prefixes of length  5:   40181
distinct prefixes of length  6:   82864
distinct prefixes of length  7:  134016
distinct prefixes of length  8:  191083
distinct prefixes of length  9:  242545
distinct prefixes of length 10:  287011


At length 10 there are about 287,000 distinct prefixes out of 400,000 games: by
ply 10 almost every opening is unique. A count-based model has data density only
up to ply 4 or 5, then it collapses.

### 1.3 Baseline C1 - positional unigram

Assigns each candidate its **marginal surprisal** at the corresponding ply and
picks the most surprising one.

The specification guarantees that the marginals are matched, so this baseline
should score around 10%. It is a control whose **expected failure** is the
information: if it scored much higher, the data generator would have an
exploitable flaw.

In [ ]:
from collections import defaultdict
pos_logp = []
for i in range(40):
    c = Counter(M[:, i]); tot = sum(c.values())
    pos_logp.append(defaultdict(lambda: -20.0,
                    {k: np.log(v / tot) for k, v in c.items()}))

sub = test_df.sample(5000, random_state=0)
S   = np.array(sub["sequence"].str.split().tolist())         # (5000,40)
yb  = sub["error_position"].to_numpy()                       # (5000,)

sc_c1 = np.empty((len(S), 10))                               # (5000,10)
for i in range(10):
    sc_c1[:, i] = [-pos_logp[i][t] for t in S[:, i]]
print("C1 positional unigram, Accuracy@1 =", np.mean(sc_c1.argmax(1) + 1 == yb))

C1 positional unigram, Accuracy@1 = 0.1314


Result: **0.130**, against the 10% of chance.

The three points of excess have a precise explanation. In isolation the signal is
null, but the argmax compares the ten candidates **within the same game**: the
nine true moves belong to a coherent game and are therefore jointly typical,
while the corrupted one is drawn from the global marginal, independent of that
game. The specification is essentially honest and there is nothing to exploit.

### 1.4 Baseline C2 - conditional surprisal on the full context

Score of candidate *i*:

$$
s_i = -\log \frac{c(m_1{:}m_i) + \alpha}{c(m_1{:}m_{i-1}) + \alpha V}
$$

where $c(\cdot)$ is the prefix count in the clean corpus and $\alpha$ is the
smoothing constant.

Using the **conditional** ratio rather than the raw prefix surprisal matters. If
the corruption is at ply 3, every prefix from 3 to 10 is rare, because they all
contain the corrupted move, so the raw surprisal would be high everywhere after
3. The conditional ratio instead is small only at step 3, because from 4 onwards
it is zero over zero and the smoothing brings the value back to uniform.

In [ ]:
ALPHA = 0.5

def raw_scores(rows10):
    out = np.empty((len(rows10), 10))                       # (N,10)
    for n, row in enumerate(rows10):
        r = tuple(row)
        for i in range(10):
            num = prefix_count[i + 1][r[:i + 1]] + ALPHA
            den = prefix_count[i][r[:i]]         + ALPHA * V
            out[n, i] = -np.log(num / den)
    return out

sc_c2 = raw_scores(S[:, :10].tolist())                      # (5000,10)
pred_c2 = sc_c2.argmax(1) + 1
print("C2 conditional surprisal, Accuracy@1 =", np.mean(pred_c2 == yb))
print("\nbreakdown by true position:")
for k in range(1, 11):
    m = yb == k
    print(f"  pos {k:2d}: {np.mean(pred_c2[m] == k):.3f}")

C2 conditional surprisal, Accuracy@1 = 0.4298

breakdown by true position:
  pos  1: 0.002
  pos  2: 0.000
  pos  3: 0.305
  pos  4: 0.410
  pos  5: 0.555
  pos  6: 0.680
  pos  7: 0.699
  pos  8: 0.645
  pos  9: 0.552
  pos 10: 0.444


Result: **0.430** overall, but the breakdown shows a rising profile, from 0.000
at positions 1 and 2 up to 0.70 at position 7, then falling.

The absolute zero at positions 1 and 2 is not weak signal, it is a **systematic
defect of the baseline**. The score $s_i$ grows monotonically with $i$ by
construction, because conditional entropy grows with the ply: at ply 1 the
numerator is a huge count (`e4` appears 219,373 times) and $s_1 \approx 0.6$,
at ply 7 the numerator is a handful of occurrences and $s_7 \approx 3$. The argmax
never picks the early positions. The scores are not comparable across positions.

### 1.5 Normalized C2 - a control that fails, and why

The natural fix: standardize per position, that is, compare the observed
surprisal with the one **expected at that ply**.

$$
\tilde s_i = \frac{s_i - \mu_i}{\sigma_i}
$$

Same logic as a z-score: a surprisal of 3 nats is enormous at ply 1 and ordinary
at ply 7. We keep this baseline because its failure is instructive.

In [ ]:
cal = clean[np.random.default_rng(1).choice(len(clean), 20000, replace=False), :10]
sc_cal = raw_scores(cal.tolist())                           # (20000,10)
mu, sd = sc_cal.mean(0), sc_cal.std(0)                      # (10,), (10,)
print("mu:", np.round(mu, 2))
print("sd:", np.round(sd, 2))

pred_c2n = ((sc_c2 - mu) / sd).argmax(1) + 1
print("\nnormalized C2, Accuracy@1 =", np.mean(pred_c2n == yb))
for k in range(1, 11):
    m = yb == k
    print(f"  pos {k:2d}: {np.mean(pred_c2n[m] == k):.3f}")

mu: [1.36 2.01 2.02 2.45 3.27 4.14 4.92 5.63 6.12 6.48]
sd: [1.3  1.07 1.41 1.65 1.9  2.05 1.98 1.72 1.47 1.17]

normalized C2, Accuracy@1 = 0.3664
  pos  1: 0.035
  pos  2: 0.014
  pos  3: 0.351
  pos  4: 0.504
  pos  5: 0.539
  pos  6: 0.599
  pos  7: 0.575
  pos  8: 0.507
  pos  9: 0.079
  pos 10: 0.457


Result: **0.366**, worse than the unnormalized 0.430, and position 9 collapses
to 0.079.

The cause is **post-corruption saturation**. If the corruption is at ply 3, the
prefix $m_1{:}m_3$ does not exist, so $c = 0$; but then every later prefix also
has $c = 0$, and for every $i \ge 4$

$$
s_i = -\log \frac{0 + \alpha}{0 + \alpha V} = \log V \approx 8.23
$$

a constant. After the corruption all scores saturate at the same maximum value.
Normalizing, that constant produces a z-score that **decreases** with $i$,
because $\mu_i$ grows. The maximum therefore falls on the first position **after**
the corruption: a systematic off-by-one error.

Conclusion: the right decision rule is not the argmax of a score, but the **first
position at which the prefix dies**.

### 1.6 Baseline C3 - first prefix break

This is the definitive non-neural baseline, the one the network is compared
against.

In [ ]:
pred_c3 = np.zeros(len(S), dtype=int)                       # (5000,)
for n in range(len(S)):
    r = tuple(S[n, :10]); hit = 0
    for i in range(1, 11):
        if prefix_count[i][r[:i]] == 0:
            hit = i; break
    if hit == 0:                                  # no break: fall back to C2
        hit = int(sc_c2[n].argmax()) + 1
    pred_c3[n] = hit

BASELINE = np.array([np.mean(pred_c3[yb == k] == k) for k in range(1, 11)])
print("C3 first prefix break, Accuracy@1 =", np.mean(pred_c3 == yb))
for k in range(1, 11):
    print(f"  pos {k:2d}: {BASELINE[k-1]:.3f}")

C3 first prefix break, Accuracy@1 = 0.4274
  pos  1: 0.000
  pos  2: 0.000
  pos  3: 0.299
  pos  4: 0.400
  pos  5: 0.551
  pos  6: 0.678
  pos  7: 0.699
  pos  8: 0.645
  pos  9: 0.552
  pos 10: 0.444


Result: **0.42**, with **0.000 at positions 1 and 2**.

These values are computed on a 5,000-example subsample of the test set, so they
fluctuate by a few thousandths between runs; the tables in Steps 8 and 8b report
an earlier draw of the same estimate.

This last figure is the most important result of Step 1, and it is a limitation
in principle rather than an implementation defect. At ply 1 only 20 moves exist
and every prefix of length 1 is extremely frequent: replacing `e4` with `d4`
produces a perfectly plausible prefix. No backward-looking statistic can detect a
corruption at the earliest plies.

**Two classes out of ten are reachable only through future context.** The
bidirectional encoder is not an optimization, it is what makes 20% of the problem
solvable at all.

| Signal source | Captured by C3 |
|---|---|
| prefix implausibility, plies 3-7 | yes, with sufficient data density |
| prefix implausibility, plies 8-10 | partially, counts become sparse |
| prefix implausibility, plies 1-2 | **no, impossible in principle** |
| inconsistency with plies 11-40 | **no, it never looks at them** |

The last two rows are the entire added value of the network.

## STEP 2 - Tokenization and data pipeline

Each SAN move is treated as an **atomic token**. There is no decomposition into
piece, file and rank, no legality check and no hand-built feature: the model has
to reconstruct the structure of the game from the data alone, as the
specification requires.

All sequences are exactly 40 tokens long, so neither padding nor masking is
needed, and the attention layers require no `attention_mask`.

Index 0 is reserved for `[UNK]`, for test-set tokens never seen in training.
Index `VOCAB` is reserved for `[MASK]`, used only in the pre-training of Step 6.

In [ ]:
vocab_list = sorted(set(M.ravel().tolist()))
stoi      = {t: i + 1 for i, t in enumerate(vocab_list)}  # 0 = [UNK]
VOCAB     = len(vocab_list) + 1                           # 3741
MASK_ID   = VOCAB                                         # 3741
VOCAB_PRE = VOCAB + 1                                     # 3742, embedding size

def encode(df):
    X = np.array([[stoi.get(t, 0) for t in r]
                  for r in df["sequence"].str.split()], dtype=np.int32)   # (N,40)
    y = df["error_position"].to_numpy().astype(np.int32) - 1              # (N,) in 0..9
    return X, y

X_all,  y_all  = encode(train_df)                         # (400000,40), (400000,)
X_test, y_test = encode(test_df)                          # ( 50000,40), ( 50000,)
print("VOCAB (incl. UNK):", VOCAB, "| VOCAB_PRE (incl. MASK):", VOCAB_PRE)
print("X_all", X_all.shape, "| X_test", X_test.shape)
print("OOV rate on the test set:", (X_test == 0).mean())

VOCAB (incl. UNK): 3741 | VOCAB_PRE (incl. MASK): 3742
X_all (400000, 40) | X_test (50000, 40)
OOV rate on the test set: 4.65e-05


### 2.1 Train / validation split

20,000 validation examples. The 50,000-example test set is never used to select
hyperparameters or checkpoints.

The permutation is seeded, so `perm` is reproducible. This matters beyond the
split itself: Step 7c uses the same `perm` to align the clean games with `X_tr`
and `X_va`.

In [ ]:
rng  = np.random.default_rng(0)
perm = rng.permutation(len(X_all))                        # (400000,)
X_va, y_va = X_all[perm[:N_VAL]], y_all[perm[:N_VAL]]     # (20000,40), (20000,)
X_tr, y_tr = X_all[perm[N_VAL:]], y_all[perm[N_VAL:]]     # (380000,40), (380000,)
print("train", X_tr.shape, "| val", X_va.shape)
print("class frequencies, train:", np.round(np.bincount(y_tr, minlength=10) / len(y_tr), 4))
print("class frequencies, val  :", np.round(np.bincount(y_va, minlength=10) / len(y_va), 4))

train (380000, 40) | val (20000, 40)
class frequencies, train: [0.0999 0.1005 0.1002 0.0995 0.1006 0.0999 0.0996 0.0998 0.0998 0.1003]
class frequencies, val  : [0.0962 0.104  0.0998 0.0952 0.102  0.1015 0.0998 0.1027 0.0997 0.099 ]


### 2.2 tf.data pipeline

400,000 x 40 integers take 64 MB, so everything fits in RAM and no generator is
needed. `tf.data` is used only for shuffling, batching and prefetching.

Labels are **one-hot** because the label smoothing used in fine-tuning requires
it.

What the calls do: `from_tensor_slices` iterates over the first dimension of the
array; `shuffle(N)` keeps a buffer of N elements to sample from; `batch` stacks
them into `(B,40)` tensors; `prefetch` lets the CPU prepare the next batch while
the GPU computes the current one.

In [ ]:
def make_ds(X, y, training):
    ds = tf.data.Dataset.from_tensor_slices((X, tf.one_hot(y, 10)))   # (40,), (10,)
    if training:
        ds = ds.shuffle(100_000, reshuffle_each_iteration=True)
    return ds.batch(BATCH).prefetch(AUTOTUNE)

ds_tr = make_ds(X_tr, y_tr, True)
ds_va = make_ds(X_va, y_va, False)
xb, yb_ = next(iter(ds_tr))
print("batch x:", xb.shape, xb.dtype, "| batch y:", yb_.shape, yb_.dtype)

batch x: (256, 40) <dtype: 'int32'> | batch y: (256, 10) <dtype: 'float32'>


## STEP 3 - Architecture

We reuse `TransformerBlock` and `TokenAndPositionEmbedding` from the course
notebook `Lab_TXT`, with one correction, and replace the head.

**Correction to the lab code.** The lab writes
`MultiHeadAttention(num_heads=h, key_dim=embed_dim)`. In Keras `key_dim` is the
**per-head** dimension, not the total one, so this inflates the cost of attention
by a factor equal to the number of heads. The correct form is `key_dim = d // h`,
which gives exactly $4d^2 + 4d$ parameters for the attention and $12d^2 + 13d$
for the whole block.

**No causal mask.** Every position must be able to read plies 11-40, which are
the only available evidence for positions 1 and 2 (see Step 1.6). A causal
decoder would be structurally unable to solve 20% of the problem.

**Pointer head.** A shared `Dense(d -> 1)` scorer applied to the first 10
positions, followed by a softmax over the 10. A `Flatten -> Dense(10)` would be
wrong for two reasons: it would cost $40 \cdot d \cdot 10 = 102{,}400$ parameters
instead of 257, and more importantly it would encode the identity of the slot in
the weights, forcing the model to learn ten times the same function "this token
is inconsistent with its context". The symmetry between positions is already
broken by the positional embedding. This is the same argument as weight sharing
in convolutions.

The per-position bias is a vector of 10 values: since the classes are uniform it
must stay close to zero, which is a free check that the model is not exploiting a
statistical shortcut.


In [ ]:
class TransformerBlock(layers.Layer):
    """Post-LN encoder block, as in Lab_TXT, with per-head key_dim."""
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1, **kw):
        super().__init__(**kw)
        self.att = layers.MultiHeadAttention(num_heads=num_heads,
                                             key_dim=embed_dim // num_heads)
        self.ffn = keras.Sequential([layers.Dense(ff_dim, activation="relu"),
                                     layers.Dense(embed_dim)])
        self.ln1 = layers.LayerNormalization(epsilon=1e-6)
        self.ln2 = layers.LayerNormalization(epsilon=1e-6)
        self.do1 = layers.Dropout(rate)
        self.do2 = layers.Dropout(rate)

    def call(self, x, training=False):                    # x: (B,40,d)
        a = self.att(x, x)                                # (B,40,d) bidirectional
        x = self.ln1(x + self.do1(a, training=training))   # (B,40,d)
        f = self.ffn(x)                                   # (B,40,d)
        return self.ln2(x + self.do2(f, training=training))


class TokenAndPositionEmbedding(layers.Layer):
    """Token and position embeddings, summed. Positions are learned, not sinusoidal."""
    def __init__(self, maxlen, vocab_size, embed_dim, **kw):
        super().__init__(**kw)
        self.tok = layers.Embedding(vocab_size, embed_dim)    # (V,d)
        self.pos = layers.Embedding(maxlen, embed_dim)        # (40,d)
        self.maxlen = maxlen

    def call(self, x):                                    # x: (B,40) int32
        return self.tok(x) + self.pos(tf.range(self.maxlen))  # (B,40,d)


class PositionScorer(layers.Layer):
    """Pointer head: one scorer shared across the first n_cand positions."""
    def __init__(self, n_cand=10, **kw):
        super().__init__(**kw)
        self.n_cand = n_cand
        self.proj = layers.Dense(1)                       # (d,1) shared
        self.bias = self.add_weight(shape=(n_cand,), initializer="zeros",
                                    name="pos_bias", trainable=True)

    def call(self, x):                                    # x: (B,40,d)
        c = x[:, :self.n_cand, :]                         # (B,10,d)
        s = tf.squeeze(self.proj(c), -1)                  # (B,10)
        return s + self.bias                              # (B,10) logits

### 3.1 Model constructors

The encoder is shared between pre-training and fine-tuning: it is literally the
same object, with different heads attached on top.

This cell also defines `zrow`, `pll_scores` and `build_joint`, used in Steps 7c,
8b and 10. They live here rather than in the step that introduces them so that
the fast path of Step 10 works without running any training step.

In [ ]:
def build_encoder():
    """Shared encoder. Vocabulary extended with the [MASK] token."""
    inp = layers.Input(shape=(MAXLEN,), dtype="int32")                  # (B,40)
    x = TokenAndPositionEmbedding(MAXLEN, VOCAB_PRE, D_MODEL)(inp)      # (B,40,d)
    for _ in range(N_BLOCKS):
        x = TransformerBlock(D_MODEL, N_HEADS, FF_DIM, DROPOUT)(x)      # (B,40,d)
    x = layers.LayerNormalization(epsilon=1e-6)(x)                      # (B,40,d)
    return keras.Model(inp, x, name="encoder")

def build_classifier(enc):
    """Encoder + pointer head. Used in Steps 5, 7 and 7b."""
    inp = layers.Input(shape=(MAXLEN,), dtype="int32")                  # (B,40)
    return keras.Model(inp, PositionScorer(10)(enc(inp)), name="classifier")

def build_joint(enc, head):
    """Two-headed model plus its inference model, with shared layers.

    The inference model reuses the SAME PositionScorer as the joint one, so after
    training it is already fitted and must not be rebuilt from scratch.
    """
    scorer = PositionScorer(10, name="pos_scorer")
    ia = layers.Input(shape=(MAXLEN,), dtype="int32", name="corrupt")   # (B,40)
    ib = layers.Input(shape=(MAXLEN,), dtype="int32", name="masked")    # (B,40)
    p  = scorer(enc(ia))                                                # (B,10)
    m  = head(enc(ib))                                                  # (B,40,3742)
    joint = keras.Model([ia, ib], {"pos": p, "mlm": m}, name="joint")

    ic = layers.Input(shape=(MAXLEN,), dtype="int32")                   # (B,40)
    infer = keras.Model(ic, scorer(enc(ic)), name="classifier_joint")
    return joint, infer


def zrow(a):
    """Per-row z-score. The decision compares the 10 candidates within the same
    game, so the legitimate normalization is per example, not global."""
    return (a - a.mean(1, keepdims=True)) / (a.std(1, keepdims=True) + 1e-8)


def pll_scores(X, bs=256):
    """Conditional surprisal of the 10 candidate positions.

    For each j, masks x_j and reads -log p(x_j | x_\\j) from the MLM head.
    10 forward passes per example, no extra parameters.
    Reads the global `scorer_mlm`, which must be assigned before calling this.
    """
    out = np.empty((len(X), 10), dtype=np.float32)              # (N,10)
    for s in range(0, len(X), bs):
        xb = tf.constant(X[s:s+bs])                             # (b,40)
        b  = tf.shape(xb)[0]
        for j in range(10):
            idx = tf.stack([tf.range(b), tf.fill([b], j)], axis=1)      # (b,2)
            xm  = tf.tensor_scatter_nd_update(xb, idx, tf.fill([b], MASK_ID))
            lg  = scorer_mlm(xm, training=False)[:, j, :]               # (b,V)
            lp  = tf.nn.log_softmax(lg, axis=-1)                        # (b,V)
            out[s:s+bs, j] = -tf.gather(lp, xb[:, j], batch_dims=1).numpy()
        if s % (bs * 20) == 0:
            print(f"  {s}/{len(X)}", end="\r")
    return out


_e = build_encoder(); _m = build_classifier(_e)
print("encoder parameters      :", _e.count_params())
print("pointer-only model      :", _m.count_params(), " (limit: 6,000,000)")
del _e, _m

encoder parameters      : 4127744
pointer-only model      : 4128011  (limit: 6,000,000)


### Parameter budget

| Component | Formula | Value |
|---|---|---|
| Token embedding | $V_{pre} \cdot d = 3742 \times 256$ | 957,952 |
| Positional embedding | $40 \cdot d$ | 10,240 |
| 4 encoder blocks | $4 (12 d^2 + 13 d)$ | 3,159,040 |
| Final LayerNorm | $2 d$ | 512 |
| Pointer head | $d + 1 + 10$ | 267 |
| **Encoder + pointer** | | **4,128,011** |
| MLM head | $d \cdot V_{pre} + V_{pre}$ | 961,694 |
| **Final model** | | **5,089,705** |

The MLM head is introduced in Step 6 for pre-training. In the first version of
this solution it was discarded after pre-training, and the delivered model had
4,128,011 parameters. Step 8b keeps it at inference time, because the
pseudo-log-likelihood it produces is a second, decorrelated score that is worth
0.9 points on the test set: 5,089,705 parameters, still under the 6,000,000 limit
with about 0.9M to spare.

With `N_BLOCKS = 5` the encoder alone would reach 4.92M and the full model 5.88M,
still admissible; `N_BLOCKS = 6` would not fit once the MLM head is kept.

## STEP 4 - Pipeline check (optional)

Before launching a long training run, we check that the model can **memorize 512
examples**. With 4.1 million parameters the capacity is far more than enough, so
the loss must drop close to zero.

If the loss stayed stuck at $\log 10 = 2.30$, there would be a bug: labels
detached from inputs, wrong indices, gradients not flowing. Better to find out in
three minutes than after half an hour of training.

Observed outcome: loss from 2.76 to **0.0017**, accuracy 1.0. The pipeline is
correct.

Set `RUN_SANITY_CHECK = True` in Step 0 to re-run it.

In [ ]:
if RUN_SANITY_CHECK:
    dbg = build_classifier(build_encoder())
    dbg.compile(optimizer=keras.optimizers.Adam(3e-4),
                loss=keras.losses.CategoricalCrossentropy(from_logits=True),
                metrics=["categorical_accuracy"])
    dbg.fit(X_tr[:512], tf.one_hot(y_tr[:512], 10),
            batch_size=64, epochs=40, verbose=2)
    del dbg
else:
    print("Step 4 disabled. Recorded outcome: loss 0.0017, accuracy 1.0.")

Step 4 disabled. Recorded outcome: loss 0.0017, accuracy 1.0.


## STEP 5 - Experiments without pre-training (optional)

These are the **ablations** that justify the pre-training of Step 6. They are not
needed to reproduce the final result, but their numbers are the reference point.

**5.a Training from scratch, 12 epochs, dropout 0.1.**
Result: val 0.9293, test 0.9312. Train/val gap of 5.6 points, with `val_loss`
rising by 62% after epoch 7 while `val_accuracy` keeps slowly improving. This is
not a contradiction: cross-entropy penalizes miscalibrated confidence, so the
model pays dearly for its errors as it becomes more certain, but the argmax stays
correct just as often. Since the official metric is Accuracy@1 and not the loss,
the checkpoint must be selected on accuracy: selecting on the loss would have
picked epoch 7 and lost 0.3 points.

**5.b Stronger regularization: dropout 0.2, 18 epochs, label smoothing.**
Result: val 0.9324, that is +0.31 points. But the train/val gap stays at 5.5
points, unchanged. Doubling the dropout and extending training by 50% changed
nothing about memorization. **Architectural regularization is not the lever**:
the limit is the amount of supervision, not how it is regularized.

**5.c Weight averaging (SWA) over three checkpoints.**
Result: 0.9312, worse than 0.9324. **Discarded.** SWA works when the snapshots
fall at different points of a flat region of the minimum, which requires a
learning rate that is still high or cyclic. Here `CosineDecay` was already at 5%
of its peak, so the three snapshots were essentially the same point: averaging
adds no diversity and only introduces a small misalignment.

The conclusion of this step is what motivates Step 6. If more regularization does
not close the gap, the missing ingredient is supervision, and the only source of
additional supervision that respects the constraints is the data itself.

Set `RUN_ABLATIONS = True` in Step 0 to re-run them. About 90 minutes.

In [ ]:
if RUN_ABLATIONS:
    # --- 5.a training from scratch ---
    m_a = build_classifier(build_encoder())
    steps = len(X_tr) // BATCH
    lr_a = keras.optimizers.schedules.CosineDecay(
        0.0, warmup_target=3e-4, warmup_steps=1000,
        decay_steps=12 * steps - 1000, alpha=0.05)
    m_a.compile(optimizer=keras.optimizers.AdamW(lr_a, weight_decay=WD, clipnorm=1.0),
                loss=keras.losses.CategoricalCrossentropy(from_logits=True),
                metrics=["categorical_accuracy"])
    m_a.fit(ds_tr, validation_data=ds_va, epochs=12, verbose=1,
            callbacks=[keras.callbacks.ModelCheckpoint(
                "abl_scratch.weights.h5", save_weights_only=True,
                monitor="val_categorical_accuracy", mode="max", save_best_only=True)])

    # --- 5.b dropout 0.2 + label smoothing, 18 epochs ---
    _d, DROPOUT = DROPOUT, 0.2
    m_b = build_classifier(build_encoder())
    lr_b = keras.optimizers.schedules.CosineDecay(
        0.0, warmup_target=3e-4, warmup_steps=1000,
        decay_steps=18 * steps - 1000, alpha=0.05)
    m_b.compile(optimizer=keras.optimizers.AdamW(lr_b, weight_decay=WD, clipnorm=1.0),
                loss=keras.losses.CategoricalCrossentropy(from_logits=True,
                                                          label_smoothing=0.05),
                metrics=["categorical_accuracy"])
    m_b.fit(ds_tr, validation_data=ds_va, epochs=18, verbose=1,
            callbacks=[keras.callbacks.ModelCheckpoint(
                "abl_drop02.weights.h5", save_weights_only=True,
                monitor="val_categorical_accuracy", mode="max", save_best_only=True)])
    DROPOUT = _d
else:
    print("Step 5 disabled. Recorded outcomes:")
    print("  5.a training from scratch     : val 0.9293, test 0.9312")
    print("  5.b dropout 0.2 + smoothing   : val 0.9324, gap unchanged")
    print("  5.c weight averaging          : val 0.9312, discarded")

Step 5 disabled. Recorded outcomes:
  5.a training from scratch     : val 0.9293, test 0.9312
  5.b dropout 0.2 + smoothing   : val 0.9324, gap unchanged
  5.c weight averaging          : val 0.9312, discarded


## STEP 6 - Self-supervised pre-training (MLM)

**The problem.** The classification task provides one label over ten classes for
each 40-token sequence: at most $\log_2 10 \approx 3.3$ bits of supervision per
example. With 380,000 examples and 4.1 million parameters the model memorizes the
training set, and Step 5 showed that no form of regularization prevents it.

**The solution.** Masked language modeling produces far denser supervision:
masking 15% of the tokens gives about 6 targets over 3742 classes per sequence,
an order of magnitude more signal, and 16 million targets in total instead of
400,000.

And it is the right task, not just any task: to reconstruct a masked token the
model has to build internally the notion of "which move is consistent with this
context", which is exactly the function the final task needs.

**Masking scheme**, BERT's: of the selected tokens, 80% become `[MASK]`, 10%
become a random token, 10% are left unchanged. Random replacement is particularly
apt here, because it is literally the situation the model will face at
fine-tuning time.

**The `sample_weight`** is the third element returned by `mlm_map`. Keras uses it
to weight the loss position by position, so reconstruction is trained only on the
masked tokens. Without it the model would learn to copy its input, which is
trivial and useless.

The data are the reconstructed clean games of Step 1.2. `correct_move` is used
only to reconstruct them, never as a model input.

In [ ]:
# ---- pre-training data: reconstructed clean games (Step 1.2) ----
X_pre_full = np.array([[stoi.get(t, 0) for t in row] for row in clean],
                      dtype=np.int32)                       # (400000, 40)

# held-out set for measuring perplexity: never seen during pre-training
X_pre     = X_pre_full[:-N_MLM_VAL]                         # (390000, 40)
X_pre_val = X_pre_full[-N_MLM_VAL:]                         # ( 10000, 40)
print("X_pre", X_pre.shape, "| X_pre_val", X_pre_val.shape)


def mlm_map(x):                                             # x: (40,) int32
    """BERT scheme: 80% [MASK], 10% random token, 10% unchanged."""
    sel = tf.random.uniform(tf.shape(x)) < MASK_PROB        # (40,) bool
    r   = tf.random.uniform(tf.shape(x))                    # (40,)
    rnd = tf.random.uniform(tf.shape(x), 1, VOCAB, dtype=tf.int32)   # (40,)
    xin = tf.where(sel & (r < 0.8), MASK_ID, x)                      # (40,)
    xin = tf.where(sel & (r >= 0.8) & (r < 0.9), rnd, xin)           # (40,)
    return xin, x, tf.cast(sel, tf.float32)   # input, target, sample_weight


def make_mlm_ds(X, shuffle):
    ds = tf.data.Dataset.from_tensor_slices(X)
    if shuffle:
        ds = ds.shuffle(100_000, reshuffle_each_iteration=True)
    return ds.map(mlm_map, num_parallel_calls=AUTOTUNE).batch(BATCH).prefetch(AUTOTUNE)


ds_pre     = make_mlm_ds(X_pre,     shuffle=True)
ds_pre_val = make_mlm_ds(X_pre_val, shuffle=False)

# ---- encoder + reconstruction head ----
# The head is a NAMED layer kept in a variable: it is needed later for the
# multi-task fine-tuning (7c) and for the PLL scoring (8b).
encoder  = build_encoder()
mlm_head = layers.Dense(VOCAB_PRE, name="mlm_head")         # (d, 3742) = 961,694 par.
mlm_in   = layers.Input(shape=(MAXLEN,), dtype="int32")     # (B, 40)
mlm      = keras.Model(mlm_in, mlm_head(encoder(mlm_in)), name="mlm")

print("pre-training parameters:", mlm.count_params(),
      "| encoder parameters:", encoder.count_params())

X_pre (390000, 40) | X_pre_val (10000, 40)
pre-training parameters: 5089438 | encoder parameters: 4127744


In [ ]:
class EncoderSnapshot(keras.callbacks.Callback):
    """Saves encoder and full MLM model at the given epochs.

    This allows several pre-training durations to be evaluated from a single run:
    instead of four separate runs of 8/12/16/20 epochs, one run of 20 with four
    save points. Step 7b then fine-tunes from each of them.
    """
    def __init__(self, enc, full, epochs):
        super().__init__()
        self.enc, self.full, self.ep = enc, full, set(epochs)

    def on_epoch_end(self, epoch, logs=None):
        e = epoch + 1
        if e in self.ep:
            self.enc.save_weights(f"encoder_mlm_e{e:02d}.weights.h5")
            self.full.save_weights(f"mlm_full_e{e:02d}.weights.h5")
            print(f"  [snapshot] encoder_mlm_e{e:02d} + mlm_full_e{e:02d}")


if RUN_PRETRAIN:
    steps = len(X_pre) // BATCH
    # decay_steps reads MLM_EPOCHS, so the schedule stretches automatically.
    lr = keras.optimizers.schedules.CosineDecay(
            0.0, warmup_target=MLM_LR, warmup_steps=1000,
            decay_steps=MLM_EPOCHS * steps - 1000, alpha=0.05)

    mlm.compile(optimizer=keras.optimizers.AdamW(lr, weight_decay=WD, clipnorm=1.0),
                loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True))

    mlm.fit(ds_pre, validation_data=ds_pre_val, epochs=MLM_EPOCHS, verbose=1,
            callbacks=[EncoderSnapshot(encoder, mlm, MLM_SNAPSHOTS)])

    # Final save. The head must be saved too: without it the PLL of Step 8b
    # cannot be reconstructed and pre-training would have to be redone.
    encoder.save_weights("encoder_mlm.weights.h5")
    mlm.save_weights("mlm_full.weights.h5")
    print("saved encoder_mlm.weights.h5 and mlm_full.weights.h5")
else:
    if os.path.exists("encoder_mlm.weights.h5"):
        encoder.load_weights("encoder_mlm.weights.h5")
        if os.path.exists("mlm_full.weights.h5"):
            mlm.load_weights("mlm_full.weights.h5")
        print("MLM weights loaded (pre-training skipped)")
    else:
        print("Step 6 disabled and no pre-training checkpoint on disk.")
        print("The encoder stays randomly initialized; Step 10 rebuilds the final")
        print("model from joint.weights.h5 and does not depend on this cell.")

Step 6 disabled and no pre-training checkpoint on disk.
The encoder stays randomly initialized; Step 10 rebuilds the final
model from joint.weights.h5 and does not depend on this cell.


### 6.1 Pre-training diagnostics

The loss printed by Keras is divided by $B \times 40$, that is by **all**
positions, while only 15% carry a non-zero weight. The printed value is therefore
about $0.15$ times the true one: a printed loss of 0.25 corresponds to roughly
1.66 nats on the tokens that are actually masked.

This cell computes the true values, on the held-out set.

In [ ]:
if os.path.exists("encoder_mlm.weights.h5"):
    # Diagnostics run on the held-out set, not on ds_pre: with 20 epochs instead of 8
    # memorization grows and numbers measured on the training set become optimistic.
    xb, yb_, wb = next(iter(ds_pre_val))                               # (B,40) x3
    lg = mlm(xb, training=False)                                       # (B,40,VOCAB_PRE)
    m  = wb > 0                                                        # (B,40) bool

    ce = keras.losses.sparse_categorical_crossentropy(yb_, lg, from_logits=True)
    print("loss on masked tokens    :", float(tf.reduce_mean(ce[m])))
    print("perplexity               :", float(tf.exp(tf.reduce_mean(ce[m]))))

    acc = tf.cast(tf.argmax(lg, -1, output_type=tf.int32) == yb_, tf.float32)
    print("reconstruction accuracy  :", float(tf.reduce_mean(acc[m])))
    print("masked tokens in batch   :", int(tf.reduce_sum(tf.cast(m, tf.int32))))
else:
    print("Step 6.1 skipped: no pre-training checkpoint on disk.")
    print("Recorded outcome: loss 1.310 nats, perplexity 3.707, accuracy 68.1%.")

Step 6.1 skipped: no pre-training checkpoint on disk.
Recorded outcome: loss 1.310 nats, perplexity 3.707, accuracy 68.1%.


Observed results, 20 epochs, held-out set: loss **1.310** nats, **perplexity
3.71**, **reconstruction accuracy 68.1%** over a vocabulary of 3742.

The first version of this notebook used 8 epochs and obtained perplexity 4.66 and
64.5% accuracy, but measured on training data; Step 7b examines the effect of the
pre-training length properly.

The model guesses the exact move about two times out of three from context alone:
it has narrowed the space of plausible moves from 3742 to fewer than four
effective candidates. It has therefore reconstructed the structure of the game
without ever being given the rules, which is the spirit of the constraint in the
specification rather than a way around it.

There is also a direct link to the final task. If the model assigns the true move
a probability corresponding to perplexity 3.71, while the corrupted move is drawn
from the marginal of that ply where plausible alternatives number in the dozens,
the likelihood ratio between true and corrupted move is typically large.
Identifying the corruption is the same quantity read differently. Step 8b makes
this link explicit and uses it directly at inference time.

## STEP 7 - Fine-tuning (baseline configuration)

The reconstruction head is discarded and the pointer head is attached to the
pre-trained encoder. This is the configuration of the first working solution; the
model that is actually delivered comes from Step 7c.

**Loss**: categorical cross-entropy with label smoothing 0.05. The target becomes
0.95 on the true class and $0.05/9$ on the others, so the gradient no longer
pushes the logits towards infinity and the model cannot gain by increasing its
confidence on examples it already gets right. This addresses exactly the
pathology observed in Step 5.a.

**No class weighting**: the classes are uniform by construction.

**Checkpoint selected on accuracy, not on the loss**, for the reason explained in
Step 5.a.

**Learning rate 1e-4**, one third of the one used without pre-training. With
already informative weights, a high learning rate destroys the representations in
the first few steps, before the new head produces sensible gradients.

The first line always reloads clean MLM weights: without it, re-running this cell
would fine-tune on top of an already converged fine-tuning, with the learning rate
schedule restarting from scratch and destroying the result.

In [ ]:
def run_finetune(enc_weights, tag, epochs=None, alpha=None, use_ema=None, seed=0):
    """Fine-tunes the pointer head on top of a pre-trained encoder.

    One degree of freedom per call. Comparison is always made on the validation
    set; the test set is never involved.
    """
    epochs  = epochs if epochs is not None else FT_EPOCHS
    alpha   = alpha  if alpha  is not None else COS_ALPHA
    use_ema = use_ema if use_ema is not None else USE_EMA

    keras.utils.set_random_seed(seed)
    encoder.load_weights(enc_weights)          # always restart from clean MLM weights
    model = build_classifier(encoder)
    print(f"[{tag}] trainable parameters:", model.count_params())

    steps = len(X_tr) // BATCH
    lr_ft = keras.optimizers.schedules.CosineDecay(
                0.0, warmup_target=FT_LR, warmup_steps=500,
                decay_steps=epochs * steps - 500, alpha=alpha)

    # ema_overwrite_frequency = steps: the weights are replaced by the EMA at the
    # end of each epoch, so validation measures the averaged weights and the
    # checkpoint selects on them. With None the EMA would only apply at fit end.
    opt = keras.optimizers.AdamW(
        lr_ft, weight_decay=WD, clipnorm=1.0,
        use_ema=use_ema, ema_momentum=EMA_MOMENTUM,
        ema_overwrite_frequency=(steps if use_ema else None))

    model.compile(
        optimizer=opt,
        loss=keras.losses.CategoricalCrossentropy(from_logits=True,
                                                  label_smoothing=LABEL_SMOOTH),
        metrics=["categorical_accuracy"])

    cbs = [keras.callbacks.ModelCheckpoint(
               f"best_{tag}.weights.h5", save_weights_only=True,
               monitor="val_categorical_accuracy", mode="max",
               save_best_only=True, verbose=0),
           keras.callbacks.EarlyStopping(
               monitor="val_categorical_accuracy", mode="max",
               patience=PATIENCE, restore_best_weights=True)]

    hist = model.fit(ds_tr, validation_data=ds_va, epochs=epochs,
                     callbacks=cbs, verbose=1)

    best = max(hist.history["val_categorical_accuracy"])
    print(f"[{tag}] best val = {best:.5f}   delta vs 0.95490 = {best-0.95490:+.5f}")
    return model, best, hist


if RUN_FINETUNE:
    model, ft_best, hist = run_finetune("encoder_mlm.weights.h5", "main")
else:
    print("Step 7 disabled. Recorded outcome: val 0.95490 at epoch 12.")

Step 7 disabled. Recorded outcome: val 0.95490 at epoch 12.


Result: **val 0.95490**, against 0.9324 without pre-training. A gain of **2.26
points**.

| | train | val | gap |
|---|---|---|---|
| without pre-training (5.b) | 0.987 | 0.932 | 5.5 points |
| with MLM pre-training | 0.983 | 0.955 | 2.9 points |

The gap halves. Pre-training did not merely speed up convergence, it **reduced
memorization**: representations are no longer built from scratch on the 380,000
labelled examples, so there is less to memorize.

A second sign pointing the same way: already at the **first epoch** the
fine-tuning reaches val 0.9313, a value that without pre-training took 17 epochs.

## STEP 7b - How much pre-training is needed

The 20-epoch pre-training of Step 6 produces four encoder snapshots. Here each of
them is fine-tuned with an identical schedule, so that the only variable is the
duration of the pre-training.

`encoder_mlm_e08_orig` is the encoder of the original 8-epoch run, included as a
reference: it is the only one whose fine-tuning is directly comparable with the
0.95490 obtained in Step 7.

Standard error on a 20,000-example validation set:
$\sqrt{0.955 \cdot 0.045 / 20000} = 0.0015$. Differences below 0.15 points are noise.

In [ ]:
if RUN_SNAPSHOT_SWEEP:
    cands = {f"e{e:02d}": f"encoder_mlm_e{e:02d}.weights.h5" for e in MLM_SNAPSHOTS}
    if os.path.exists("encoder_mlm_e08_orig.weights.h5"):
        cands["e08_orig"] = "encoder_mlm_e08_orig.weights.h5"   # 0.95490 reference

    sweep = {}
    for tag, w in cands.items():
        if not os.path.exists(w):
            print(f"missing {w}, skipping"); continue
        _, sweep[tag], _ = run_finetune(w, tag)

    print("\nsnapshot -> fine-tuning val accuracy")
    for t, v in sweep.items():
        print(f"  {t:9s}: {v:.5f}   delta vs 0.95490: {v-0.95490:+.5f}")

    if sweep:
        best = max(sweep, key=sweep.get)
        print(f"\nbest: {best}, val {sweep[best]:.5f}")
        shutil.copy(cands[best], "encoder_mlm.weights.h5")
        mlm_src = cands[best].replace("encoder_mlm", "mlm_full")
        if os.path.exists(mlm_src):
            shutil.copy(mlm_src, "mlm_full.weights.h5")
            print("reference weights updated, PLL available")
        else:
            print(f"{mlm_src} not found: PLL not available with this snapshot")
else:
    print("Step 7b disabled. Recorded outcome: 20 epochs is best, val 0.96000.")

Step 7b disabled. Recorded outcome: 20 epochs is best, val 0.96000.


| snapshot | fine-tuning val | annealing |
|---|---|---|
| 8 epochs, original run | 0.9535 | complete |
| 8 epochs, truncated | 0.9560 | incomplete |
| 12 epochs, truncated | 0.9582 | incomplete |
| 16 epochs, truncated | 0.9578 | incomplete |
| **20 epochs** | **0.9600** | complete |

**Two readings, one of which is a caveat.**

The clean comparison is among the 8, 12 and 16 snapshots, which share the same
condition of incomplete annealing: 8 to 12 gains 0.22 points, 12 to 16 loses 0.04,
which is noise. **The benefit of longer pre-training saturates between 12 and 16
epochs.** A 32-epoch run was therefore not attempted.

The further jump of the 20-epoch snapshot comes together with the complete
annealing of the cosine schedule, so the two effects are confounded and this
design cannot separate them. The defensible statement is that a longer and fully
annealed pre-training takes the result from 0.9535 to 0.9600, not that 20 epochs
beat 12.

**Measured run-to-run variance.** The 8-epoch reference, re-run in the same
session with the same seed and the same schedule, gave 0.95350 instead of the
0.95490 of Step 7. Those 0.14 points are the empirical noise floor, consistent
with the theoretical standard error of 0.0015. Every delta reported from here on
is read against that yardstick.

## STEP 7c - Multi-task fine-tuning: pointer + continued MLM (final model)

Ordinary fine-tuning specializes the encoder on the 3.3-bit task and gradually
destroys the reconstruction ability learned during pre-training. Two consequences:

1. the encoder goes back to memorizing, which is the residual 2.9-point gap;
2. the MLM head becomes unusable at inference, which rules out the PLL scoring of
   Step 8b.

Keeping the MLM loss active during fine-tuning lets a single encoder serve both
heads. The parameter count is 4,128,011 + 961,694 = 5,089,705, under the
6,000,000 limit.

The two inputs are the corrupted sequence, for the pointer head, and the
corresponding clean game with random masking, for the MLM head. `correct_move`
only enters through the reconstruction of the clean games in Step 1.2, never as a
model input.

**The hypothesis behind this step is only half confirmed.** The gap does not
close: 2.6 points against the 2.5 of plain fine-tuning, and validation accuracy
is 0.9594 against 0.9600, a difference below the measured noise floor. What does
work is the second point: the MLM head stays aligned with the encoder, and that
is what makes Step 8b possible. Step 7d quantifies the trade-off directly.

In [ ]:
# X_pre_full follows the order of train_df, the same as X_all, so the same perm
# from Step 2.1 yields clean games aligned with X_tr and X_va.
Xc_tr = X_pre_full[perm[N_VAL:]]          # (380000, 40) clean, train
Xc_va = X_pre_full[perm[:N_VAL]]          # ( 20000, 40) clean, validation
assert len(Xc_tr) == len(X_tr) and len(Xc_va) == len(X_va)


def make_joint_ds(Xcorr, y, Xclean, training):
    """Yields ((corrupted, masked), {pos, mlm}, {sample weights})."""
    a = tf.data.Dataset.from_tensor_slices((Xcorr, tf.one_hot(y, 10)))   # (40,),(10,)
    b = tf.data.Dataset.from_tensor_slices(Xclean).map(
            mlm_map, num_parallel_calls=AUTOTUNE)                        # 3x (40,)
    ds = tf.data.Dataset.zip((a, b)).map(
        lambda A, B: ((A[0], B[0]),
                      {"pos": A[1], "mlm": B[1]},
                      {"pos": 1.0, "mlm": B[2]}),
        num_parallel_calls=AUTOTUNE)
    if training:
        ds = ds.shuffle(50_000, reshuffle_each_iteration=True)
    return ds.batch(BATCH).prefetch(AUTOTUNE)


CKPT_JOINT = f"joint_lambda{LAMBDA_MLM}.weights.h5"

if RUN_JOINT_FT:
    keras.utils.set_random_seed(0)
    encoder.load_weights("encoder_mlm.weights.h5")
    mlm.load_weights("mlm_full.weights.h5")        # also restores mlm_head

    jm, jm_infer = build_joint(encoder, mlm_head)
    # jm_infer is the pointer path alone. The delivered model also keeps the MLM
    # head for Step 8b, hence 5,089,705 parameters in total.
    print("parameters, both heads :", jm.count_params())        # 5,089,705
    print("parameters, pointer    :", jm_infer.count_params())  # 4,128,011

    steps = len(X_tr) // BATCH
    lr_j = keras.optimizers.schedules.CosineDecay(
               0.0, warmup_target=FT_LR, warmup_steps=500,
               decay_steps=FT_EPOCHS * steps - 500, alpha=COS_ALPHA)

    jm.compile(
        optimizer=keras.optimizers.AdamW(lr_j, weight_decay=WD, clipnorm=1.0),
        loss={"pos": keras.losses.CategoricalCrossentropy(
                         from_logits=True, label_smoothing=LABEL_SMOOTH),
              "mlm": keras.losses.SparseCategoricalCrossentropy(from_logits=True)},
        loss_weights={"pos": 1.0, "mlm": LAMBDA_MLM},
        metrics={"pos": ["categorical_accuracy"]})

    hist_j = jm.fit(
        make_joint_ds(X_tr, y_tr, Xc_tr, training=True),
        validation_data=make_joint_ds(X_va, y_va, Xc_va, training=False),
        epochs=FT_EPOCHS, verbose=1,
        callbacks=[keras.callbacks.ModelCheckpoint(
                       CKPT_JOINT, save_weights_only=True,
                       monitor="val_pos_categorical_accuracy", mode="max",
                       save_best_only=True, verbose=1),
                   keras.callbacks.EarlyStopping(
                       monitor="val_pos_categorical_accuracy", mode="max",
                       patience=PATIENCE, restore_best_weights=True)])

    bj = max(hist_j.history["val_pos_categorical_accuracy"])
    print(f"[joint lambda={LAMBDA_MLM}] best val = {bj:.5f}")
    print(f"  val_mlm_loss at last epoch = {hist_j.history['val_mlm_loss'][-1]:.4f}")

    # The delivered configuration is lambda = 0.3; its checkpoint becomes the
    # canonical one loaded by Step 10.
    if LAMBDA_MLM == 0.3:
        shutil.copy(CKPT_JOINT, "joint.weights.h5")
        print("copied to joint.weights.h5")
else:
    print("Step 7c disabled. Recorded outcome: val 0.95940, val_mlm_loss 0.2051.")

Step 7c disabled. Recorded outcome: val 0.95940, val_mlm_loss 0.2051.


Result: **val 0.95940**, against 0.96000 for plain fine-tuning. The difference of
0.06 points is below the 0.14-point noise floor measured in Step 7b: the two
configurations are indistinguishable on the pointer task alone.

`val_mlm_loss` ends at 0.2051, against the 0.1908 reached at the end of
pre-training, that is perplexity rising from 3.71 to about 3.90. With
$\lambda = 0.3$ the auxiliary loss slows the drift of the encoder without
stopping it.

This step is kept in the pipeline not for the number above but because it is the
**structural prerequisite** of Step 8b: after ordinary fine-tuning the MLM head is
no longer aligned with the encoder, and using two separate encoders would cost
9.2M parameters, outside the budget.

## STEP 7d - The lambda trade-off (discarded)

Step 7c leaves an obvious question open. If $\lambda = 0.3$ slows the drift of
the encoder without stopping it, a larger $\lambda$ should preserve the language
model better, and Step 8b shows that a better language model is worth almost a
point. The experiment was run with $\lambda = 1.0$, everything else identical.

**The mechanism works exactly as predicted, and the net effect is zero.**

| | lambda 0.3 | lambda 1.0 |
|---|---|---|
| `val_mlm_loss` | 0.2051 | **0.1951** |
| corresponding perplexity | ~3.90 | **~3.71** |
| pointer alone | 0.9594 | 0.9583 |
| PLL alone | 0.9559 | **0.9574** |
| best gamma (Step 8b) | 2.0 | 3.0 |
| **combination** | **0.9682** | 0.9688 |

Every intermediate link is confirmed. A larger $\lambda$ brings `val_mlm_loss`
back to its end-of-pre-training value, the PLL score improves by 0.15 points, and
the optimal $\gamma$ rises, meaning the combination puts more trust in a score
that has become more reliable.

But the pointer loses 0.11 points, and the two effects cancel: the combination
moves by 0.06 points, again below the noise floor. **$\lambda$ is not a
hyperparameter to be optimized, it is a slider on a nearly zero-sum trade-off**
between the two scores that share the encoder. There is no reason to expect
$\lambda = 2$ or 3 to break the symmetry, and pushing further would degrade the
pointer more than it improves the PLL.

The delivered model therefore keeps $\lambda = 0.3$.

To reproduce: set `LAMBDA_MLM = 1.0` and `RUN_JOINT_FT = True` in Step 0 and re-run
Step 7c. About 40 minutes. The checkpoint is written to a separate file, so the
delivered model is not overwritten.

In [ ]:
# Compares the two lambda values on the validation set. Requires both checkpoints;
# it only runs inference, no training.
def eval_lambda(ckpt, gammas=(1.0, 2.0, 3.0, 4.0)):
    global scorer_mlm
    jm_l, jm_inf = build_joint(build_encoder(), layers.Dense(VOCAB_PRE, name="mlm_head"))
    jm_l.load_weights(ckpt)
    enc_l  = jm_l.get_layer("encoder")
    head_l = jm_l.get_layer("mlm_head")
    inp    = layers.Input(shape=(MAXLEN,), dtype="int32")            # (B,40)
    scorer_mlm = keras.Model(inp, head_l(enc_l(inp)))

    s_ptr = jm_inf.predict(X_va, batch_size=512, verbose=0)          # (20000,10)
    z     = zrow(pll_scores(X_va))                                   # (20000,10)
    print(f"\n{ckpt}")
    print("  pointer alone:", float(np.mean(s_ptr.argmax(1) == y_va)))
    print("  PLL alone    :", float(np.mean(z.argmax(1) == y_va)))
    for g in gammas:
        print(f"  gamma={g:4.1f}   : {float(np.mean((s_ptr + g*z).argmax(1) == y_va)):.5f}")


CKPTS = ["joint_lambda0.3.weights.h5", "joint_lambda1.0.weights.h5"]

if all(os.path.exists(c) for c in CKPTS):
    for c in CKPTS:
        eval_lambda(c)
else:
    print("Step 7d skipped: both lambda checkpoints are required.")
    print("Recorded outcome: val_mlm_loss 0.2051 -> 0.1951, PLL alone 0.9559 -> 0.9574,")
    print("combination 0.9682 -> 0.9688, below the noise floor. lambda = 0.3 kept.")

Step 7d skipped: both lambda checkpoints are required.
Recorded outcome: val_mlm_loss 0.2051 -> 0.1951, PLL alone 0.9559 -> 0.9574,
combination 0.9682 -> 0.9688, below the noise floor. lambda = 0.3 kept.


## STEP 8 - Test set evaluation (pointer only)

Evaluation on the full 50,000-example test set, using the `accuracy_at_1`
function given in the specification, applied to the softmax probabilities.

This step measures the pointer head alone, which is the baseline configuration of
Step 7. The delivered result is in Step 8b.

It requires `model`, produced by Step 7 with `RUN_FINETUNE = True`; otherwise the
recorded numbers below apply.

In [ ]:
if "model" in globals():
    logits = model.predict(X_test, batch_size=512, verbose=1)      # (50000,10)
    probs  = tf.nn.softmax(logits).numpy()                         # (50000,10)
    yt     = test_df["error_position"].to_numpy()                  # (50000,)

    print("\nACCURACY@1 (official metric):", accuracy_at_1(yt, probs))

    pred = probs.argmax(1) + 1
    print("\nbreakdown by position, network vs C3 baseline:")
    for k in range(1, 11):
        m = yt == k
        print(f"  pos {k:2d}:  network {np.mean(pred[m]==k):.3f}   baseline {BASELINE[k-1]:.3f}")

    b = [w for w in model.trainable_weights if "pos_bias" in w.name][0].numpy()
    print("\npos_bias:", np.round(b, 3), "  (uniform classes -> expected near 0)")
else:
    print("Step 8 skipped: `model` requires Step 7 with RUN_FINETUNE = True.")
    print("Recorded outcome: Accuracy@1 = 0.95652 on the test set.")

Step 8 skipped: `model` requires Step 7 with RUN_FINETUNE = True.
Recorded outcome: Accuracy@1 = 0.95652 on the test set.


### Recorded results

**Accuracy@1 on the test set: 0.95652** (pointer only, 8-epoch pre-training)

| pos | network | C3 baseline |
|---|---|---|
| 1 | 0.979 | 0.000 |
| 2 | 0.968 | 0.002 |
| 3 | 0.953 | 0.303 |
| 4 | 0.956 | 0.400 |
| 5 | 0.949 | 0.553 |
| 6 | 0.951 | 0.680 |
| 7 | 0.949 | 0.701 |
| 8 | 0.949 | 0.645 |
| 9 | 0.955 | 0.546 |
| 10 | 0.957 | 0.421 |

The most telling comparison is in the first two rows. The positions on which the
count-based baseline is **exactly blind** are the ones on which the network does
**best of all**. This inversion is the direct demonstration of the project's
thesis: at ply 1 the opening is so constrained that the first move is strongly
determined by the following 39 plies, and only a bidirectional encoder can read
that evidence.

The profile is then nearly flat between 0.949 and 0.979, slightly lower towards
the middle. This is consistent with the rising entropy measured in Step 1.1: the
later in the opening, the more plausible moves exist, and the more cases in which
the substitution is genuinely indistinguishable.

The `pos_bias` vector stays within 0.011 in absolute value: the model has learned
no positional prior, as it should with uniform classes. Predictions come from the
content of the sequence, not from a statistical shortcut.

## STEP 8b - Pseudo-log-likelihood scoring (final result)

The pointer head produces one score per position. A second, structurally
different score is obtained by masking each candidate and reading the surprisal
of the MLM head at full bidirectional context:

$$
\sigma_j = -\log p_\theta(x_j \mid x_{\setminus j}), \qquad j = 1,\ldots,10
$$

$$
s_j^{\text{final}} = s_j^{\text{pointer}} + \gamma \cdot \tilde\sigma_j
$$

where $\tilde\sigma$ is the per-row z-score. The decision compares the ten
candidates within the same game, so the legitimate normalization is per example,
not global.

This is the neural version of the C2 baseline of Step 1.4, which scored 0.430 and
failed only because of count sparsity: by ply 10 almost every opening is unique,
while the MLM encoder has narrowed the space of plausible moves to fewer than four
effective candidates.

This step combines two components that are already part of this notebook: the MLM
head trained in Step 6, whose masked-reconstruction objective follows the same
scheme used in the course notebook `SST_reco_VIT_demo` for obfuscated image
regions, and the conditional-surprisal decision rule of the C2 baseline in Step
1.4. The count-based version of that rule failed only because of data sparsity;
here the same quantity is estimated by the neural language model instead.

**Masking is mandatory.** Without it the residual stream carries $x_j$ all the way
to the last layer, the head copies it, and $\sigma_j$ becomes uninformative.

This step requires Step 7c: after ordinary fine-tuning the MLM head is no longer
aligned with the encoder. Keeping the head at inference time costs 961,694
parameters, bringing the total to 5,089,705, and 10 extra forward passes per
example. The specification sets no constraint on inference speed.

In [ ]:
if RUN_PLL:
    jm, jm_infer = build_joint(encoder, mlm_head)
    jm.load_weights("joint.weights.h5")
    # Encoder already fine-tuned in multi-task mode, MLM head still aligned.
    scorer_mlm = keras.Model(mlm_in, mlm_head(encoder(mlm_in)))

    # ---- gamma tuning on the VALIDATION set ----
    s_ptr_va = jm_infer.predict(X_va, batch_size=512, verbose=0)    # (20000,10)
    z_va     = zrow(pll_scores(X_va))                               # (20000,10)

    print("\npointer alone (gamma = 0)   :", float(np.mean(s_ptr_va.argmax(1) == y_va)))
    print("PLL alone (gamma -> infinity):", float(np.mean(z_va.argmax(1) == y_va)))
    print("\ngamma -> validation accuracy")
    for g in GAMMA_GRID:
        v = float(np.mean((s_ptr_va + g * z_va).argmax(1) == y_va))
        print(f"  gamma={g:5.2f}  val={v:.5f}")
    print(f"\nselected: gamma = {GAMMA_BEST} (see the discussion below)")

    # ---- applied to the TEST set, once ----
    s_ptr_te = jm_infer.predict(X_test, batch_size=512, verbose=1)  # (50000,10)
    z_te     = zrow(pll_scores(X_test))                             # (50000,10)
    probs_te = tf.nn.softmax(s_ptr_te + GAMMA_BEST * z_te).numpy()  # (50000,10)
    yt = test_df["error_position"].to_numpy()                       # (50000,)

    print("\nACCURACY@1 with PLL   :", accuracy_at_1(yt, probs_te))
    print("ACCURACY@1 pointer only:",
          accuracy_at_1(yt, tf.nn.softmax(s_ptr_te).numpy()))

    pred = probs_te.argmax(1) + 1
    print("\nbreakdown by position:")
    for k in range(1, 11):
        m = yt == k
        print(f"  pos {k:2d}:  network {np.mean(pred[m]==k):.3f}   baseline {BASELINE[k-1]:.3f}")

    b = [w for w in jm_infer.trainable_weights if "pos_bias" in w.name][0].numpy()
    print("\npos_bias:", np.round(b, 3))
    print("max |pos_bias|:", np.abs(b).max(), " (uniform classes -> expected near 0)")
else:
    print("Step 8b disabled. Recorded outcome: Accuracy@1 = 0.96894 with gamma = 2.0.")

Step 8b disabled. Recorded outcome: Accuracy@1 = 0.96894 with gamma = 2.0.


### Recorded results

**Accuracy@1 on the test set: 0.96894**, against 0.95954 for the pointer alone on
the same model. A gain of **0.94 points**, about seven times the measured noise
floor.

| gamma | validation |
|---|---|
| 0.0 (pointer alone) | 0.9594 |
| 0.5 | 0.9635 |
| 1.0 | 0.9665 |
| 1.5 | 0.9679 |
| **2.0** | **0.9682** |
| 3.0 | 0.9682 |
| 5.0 | 0.9670 |
| 10.0 | 0.9636 |
| infinity (PLL alone) | 0.9559 |

**The central result is in the two extremes.** Pointer alone 0.9594, PLL alone
0.9559, combination 0.9682. Two scores of almost identical strength taken
separately, yet the combination gains 1.2 points over the better of the two. The
errors of the two predictors are largely decorrelated, and it is that, rather than
the individual strength of either, that produces the gain. This is an ensemble at
almost zero parameter cost, which is why it is preferable to training two models:
two independent encoders would cost 9.2M parameters and fall outside the budget.

$\gamma = 2.0$ is chosen over $\gamma = 3.0$, which scores identically on
validation, because it sits further from the region where the curve collapses and
is therefore less sensitive to a mismatch between validation and test. The choice
is confirmed after the fact: the test gives 0.96894, slightly **above** the
validation value, so $\gamma$ is not overfitted.

| pos | network | C3 baseline |
|---|---|---|
| 1 | 0.983 | 0.000 |
| 2 | 0.974 | 0.002 |
| 3 | 0.957 | 0.303 |
| 4 | 0.964 | 0.400 |
| 5 | 0.959 | 0.553 |
| 6 | 0.965 | 0.680 |
| 7 | 0.963 | 0.701 |
| 8 | 0.970 | 0.645 |
| 9 | 0.976 | 0.546 |
| 10 | 0.978 | 0.421 |

Compared with the pointer-only model of Step 8 the profile has changed shape: it
was flat with a minimum in the middle, it now rises towards the right, from 0.957
at ply 3 to 0.978 at ply 10. The PLL helps the late positions most, where the past
context is richer and where the count-based baseline collapsed from sparsity. The
minimum moves to ply 3, where the opening is still tightly constrained but the
prefix is too short for conditional surprisal to discriminate.

`pos_bias` stays within **0.0098** in absolute value, as in the pointer-only
model: no positional prior has been learned.

## STEP 8c - Post-hoc calibration (no gain)

The combination of Step 8b uses a single $\gamma$ and the raw pointer logits. Two
natural refinements were tested, both free: they operate on scores already
computed and require no training.

**One gamma per position.** The breakdown of Step 8b shows that the PLL helps the
late positions much more than ply 3, so a single weight is a compromise. Ten
weights were fitted by coordinate ascent on the validation set.

**Temperature on the pointer.** The two scores are summed on different scales.
Rescaling the pointer logits by $1/T$ before the sum adds one degree of freedom
next to $\gamma$, and is the standard way of recalibrating a classifier whose
confidence is miscalibrated, which Step 5.a showed to be the case here.

Acceptance criteria were fixed **before** looking at the numbers: at least +0.2
points over 0.96820 for the ten-parameter variant, which is the one most exposed
to overfitting the validation set, and +0.15 for the single extra parameter.

In [ ]:
# Requires s_ptr_va and z_va, produced by Step 8b with RUN_PLL = True.
if "z_va" in globals():

    # ---- one gamma per position, coordinate ascent ----
    g_pos = np.full(10, GAMMA_BEST)                                  # (10,)
    for _ in range(3):
        for j in range(10):
            best_g, best_v = g_pos[j], -1.0
            for g in [0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 4.0, 5.0]:
                t = g_pos.copy(); t[j] = g
                v = float(np.mean((s_ptr_va + t * z_va).argmax(1) == y_va))
                if v > best_v: best_g, best_v = g, v
            g_pos[j] = best_g
    v_pos = float(np.mean((s_ptr_va + g_pos * z_va).argmax(1) == y_va))
    print("per-position gamma:", np.round(g_pos, 2))
    print(f"val: {v_pos:.5f}   (reference, single gamma = {GAMMA_BEST}: 0.96820)")

    # ---- temperature on the pointer ----
    print("\ntemperature x gamma:")
    for T in [0.5, 0.75, 1.0, 1.5, 2.0]:
        row = [f"g={g}: {float(np.mean((s_ptr_va/T + g*z_va).argmax(1) == y_va)):.5f}"
               for g in [1.5, 2.0, 2.5, 3.0]]
        print(f"  T={T:4.2f}  " + "  ".join(row))

else:
    print("Step 8c skipped: requires the validation scores from Step 8b.")
    print("Recorded outcome: per-position gamma 0.96880, best temperature T = 1.")
    print("Both refinements below the acceptance threshold.")

Step 8c skipped: requires the validation scores from Step 8b.
Recorded outcome: per-position gamma 0.96880, best temperature T = 1.
Both refinements below the acceptance threshold.


### Recorded results

**Both refinements are null.**

Per-position gamma reaches 0.96880 against 0.96820, that is +0.06 points, well
under the +0.2 threshold. The fitted weights are themselves the argument: eight
out of ten stay at 2.0 and the other two move by half a unit. A procedure with ten
free parameters fitted on 20,000 examples essentially rediscovered the uniform
value, so the +0.06 is optimization noise rather than structure.

| T | gamma 1.5 | gamma 2.0 | gamma 2.5 | gamma 3.0 |
|---|---|---|---|---|
| 0.50 | 0.9651 | 0.9665 | 0.9674 | 0.9679 |
| 0.75 | 0.9672 | 0.9679 | 0.9682 | 0.9682 |
| **1.00** | 0.9679 | **0.9682** | 0.9682 | 0.9682 |
| 1.50 | 0.9682 | 0.9682 | 0.9678 | 0.9675 |
| 2.00 | 0.9682 | 0.9677 | 0.9670 | 0.9665 |

The temperature surface is a plateau whose maximum is exactly $T = 1$, degrading
in every direction.

This is informative rather than disappointing. It says that $\gamma = 2.0$ at unit
temperature is already the optimum of the linear combination of the two scores,
and that the remaining error is not recoverable by post-processing. **The limit
lies in the two scores, not in the way they are combined**, which is what led to
Step 7d testing whether the PLL score itself could be improved.

## STEP 9 - Discarded experiment: longer fine-tuning

At the end of Step 7 the `val_accuracy` was still rising at the last epoch, which
suggested extending training. A 24-epoch run, identical in everything else, gave
**0.9525 against 0.9549**: worse.

The reason is the **learning rate schedule**. `CosineDecay` spreads the decay over
the whole duration of the run, so:

- with 12 epochs the learning rate drops quickly and is almost zero in the last
  steps: the model settles into the minimum and validation keeps rising to the end;
- with 24 epochs the decay is diluted, and at epoch 13 the learning rate is still
  about 45% of its peak: the model keeps bouncing around a wide region instead of
  settling, and memorizes in the meantime (train accuracy 0.993 against 0.983).

The two curves are in fact identical up to epoch 6 and diverge only afterwards.

**Operating rule: `FT_EPOCHS` and `COS_ALPHA` must be moved together.** Extending
fine-tuning without lowering `COS_ALPHA` dilutes the final annealing, which was the
real reason the 12-epoch run worked.

The same rule explains a detail of Step 7b: the intermediate snapshots are
truncations of a 20-epoch cosine, not equivalents of shorter runs, which is why
the comparison there is restricted to snapshots sharing the same annealing state.

Set `RUN_LONG_FT = True` in Step 0 to re-run it.

In [ ]:
if RUN_LONG_FT:
    encoder.load_weights("encoder_mlm.weights.h5")
    m_long = build_classifier(encoder)
    steps  = len(X_tr) // BATCH
    lr_l = keras.optimizers.schedules.CosineDecay(
               0.0, warmup_target=FT_LR, warmup_steps=500,
               decay_steps=24 * steps - 500, alpha=0.02)
    m_long.compile(
        optimizer=keras.optimizers.AdamW(lr_l, weight_decay=WD, clipnorm=1.0),
        loss=keras.losses.CategoricalCrossentropy(from_logits=True,
                                                  label_smoothing=LABEL_SMOOTH),
        metrics=["categorical_accuracy"])
    m_long.fit(ds_tr, validation_data=ds_va, epochs=24, verbose=1,
               callbacks=[keras.callbacks.ModelCheckpoint(
                   "long_ft.weights.h5", save_weights_only=True,
                   monitor="val_categorical_accuracy", mode="max", save_best_only=True),
                   keras.callbacks.EarlyStopping(
                   monitor="val_categorical_accuracy", mode="max",
                   patience=5, restore_best_weights=True)])
else:
    print("Step 9 disabled. Recorded outcome: val 0.9525, worse than 0.9549.")

Step 9 disabled. Recorded outcome: val 0.9525, worse than 0.9549.


## STEP 10 - Loading the trained weights

Fast path to reproduce the final result **without training**. Requires Steps 0,
1.1, 1.2, 2, 2.1, 3 and 3.1, which take about five minutes in total.

In [ ]:
# ============ STEP 10 - Loading the trained weights ============
# Fast path: reproduces the final result without training.
# Requires Steps 0, 1.1, 1.2, 2, 2.1, 3, 3.1.

CKPT = "joint.weights.h5"

# Trained weights, published as a GitHub Release asset (61.3 MB)
WEIGHTS_URL = "https://github.com/gabrielmario-tavini/chess-move-error-detection/releases/download/v1.0/joint.weights.h5"

if not os.path.exists(CKPT):
    # On Kaggle the weights may be attached as a dataset; browser downloads replace
    # dots with underscores, so both spellings are searched for. Otherwise they are
    # downloaded from the GitHub Release.
    hits = (glob.glob("/kaggle/input/**/joint.weights.h5", recursive=True) +
            glob.glob("/kaggle/input/**/joint_weights.h5",  recursive=True))
    if hits:
        shutil.copy(hits[0], CKPT)
        print("copied from", hits[0])
    else:
        import urllib.request
        print("downloading", WEIGHTS_URL)
        urllib.request.urlretrieve(WEIGHTS_URL, CKPT)

assert os.path.exists(CKPT), "weights not found"
print(f"{os.path.getsize(CKPT)/1e6:.1f} MB  {CKPT}   (expected 61.3 MB)")

# ---- rebuilding the final model ----
encoder_l  = build_encoder()
mlm_head_l = layers.Dense(VOCAB_PRE, name="mlm_head")
jm_l, jm_infer_l = build_joint(encoder_l, mlm_head_l)
jm_l.load_weights(CKPT)
print("PARAMETERS:", jm_l.count_params(), " (limit: 6,000,000)")

# ---- combined scoring ----
# scorer_mlm is the global read by pll_scores: it must be reassigned here,
# otherwise the encoder left in memory would be evaluated instead of the one
# just loaded from disk.
mlm_in_l   = layers.Input(shape=(MAXLEN,), dtype="int32")            # (B,40)
scorer_mlm = keras.Model(mlm_in_l, mlm_head_l(encoder_l(mlm_in_l)))

s_ptr = jm_infer_l.predict(X_test, batch_size=512, verbose=1)        # (50000,10)
s_pll = pll_scores(X_test)                                           # (50000,10)
probs = tf.nn.softmax(s_ptr + GAMMA_BEST * zrow(s_pll)).numpy()      # (50000,10)

print("\nACCURACY@1:", accuracy_at_1(test_df["error_position"].to_numpy(), probs))

61.3 MB  joint.weights.h5   (expected 61.3 MB)
PARAMETERS: 5089705  (limit: 6,000,000)
98/98 ━━━━━━━━━━━━━━━━━━━━ 9s 64ms/step

ACCURACY@1: 0.96894


Correctness check: if `count_params()` prints **5,089,705** and the accuracy is
**0.96894**, the weights are the right ones. If the accuracy were around 0.10 the
load failed silently because the architecture built here does not match the saved
one: check `D_MODEL`, `N_BLOCKS`, `N_HEADS` and `FF_DIM` in Step 0.

## STEP 11 - Summary

### Progression of results

| Model | Val | Test | Notes |
|---|---|---|---|
| random classifier | - | 0.100 | 10 uniform classes |
| C1, positional unigram | - | 0.130 | confirms the marginals are matched |
| C2, conditional surprisal | - | 0.430 | blind on positions 1 and 2 |
| normalized C2 | - | 0.366 | worse, post-corruption saturation |
| **C3, first prefix break** | - | **0.426** | reference non-neural baseline |
| transformer, no pre-training | 0.9293 | 0.9312 | train/val gap 5.6 points |
| transformer, dropout 0.2 + smoothing | 0.9324 | - | gap unchanged |
| transformer + SWA | 0.9312 | - | discarded |
| transformer + MLM pre-training, 8 epochs | 0.9549 | 0.9565 | gap 2.9 points |
| transformer + MLM, 24-epoch fine-tuning | 0.9525 | - | discarded, diluted annealing |
| transformer + MLM pre-training, 20 epochs | 0.9600 | - | +0.65 over the re-run reference |
| + multi-task fine-tuning, lambda 0.3 | 0.9594 | 0.9595 | no direct gain, enables the PLL |
| + multi-task, lambda 1.0 | 0.9583 | - | zero-sum trade-off, discarded |
| + post-hoc calibration | 0.9688 | - | below threshold, discarded |
| **+ PLL scoring, gamma 2.0** | **0.9682** | **0.9689** | **delivered model** |

**A note on variance.** The 8-epoch reference, re-run in the same session with the
same seed and the same schedule, gave 0.95350 instead of 0.95490. Those 0.14 points
are the measured run-to-run variance, consistent with the theoretical standard
error $\sqrt{0.955 \cdot 0.045 / 20000} = 0.0015$ on a 20,000-example validation
set. Every delta in the table is read against that yardstick: differences below
0.15 points are not distinguishable from noise.

### Delivered model

- **Bidirectional** transformer encoder, 4 blocks, `d_model = 256`, 8 heads,
  FFN 1024, dropout 0.1
- **Pointer** head with a scorer shared across the 10 candidate positions
- **MLM** head kept at inference time to compute the PLL
- **MLM pre-training**, 20 epochs, 15% masking, on the provided training games only
- **Multi-task fine-tuning**, 12 epochs, lambda = 0.3
- **Combined scoring** with gamma = 2.0
- **Trainable parameters: 5,089,705** out of a 6,000,000 limit
  (encoder 4,127,744 + pointer head 267 + MLM head 961,694)
- Inference: 11 forward passes per example, 1 for the pointer and 10 for the PLL
- **Accuracy@1 on the 50,000-example public test set: 0.96894**

### The four choices that determine the result

**Bidirectional encoder.** The specification includes plies 11-40 in the input
because they are the continuation of the original game, hence inconsistent with the
corrupted move. Step 1.6 shows that for positions 1 and 2 this is the only evidence
that exists: a backward-looking baseline scores exactly 0.000 on those classes,
while the network reaches 0.983 and 0.974.

**Pointer head with a shared scorer.** The output is not an abstract class but a
position in the sequence. A shared scorer costs 257 parameters instead of 102,400
and, more importantly, does not force the model to learn the same function ten
times over. The symmetry between positions is already broken by the positional
embedding. This is the same argument as weight sharing in convolutions.

**Self-supervised pre-training.** The task provides 3.3 bits of supervision per
sequence, far too few for 4.1M parameters, and Step 5 shows that regularization
cannot compensate. MLM raises supervision to 16 million targets and teaches the
model exactly the notion it needs, namely which move is consistent with a context.
Overall gain: from 0.9324 to 0.9600.

**Combined pointer + PLL scoring.** A second score, obtained by masking each
candidate and reading the conditional surprisal, is almost as strong as the pointer
on its own but makes different mistakes: pointer alone 0.9594, PLL alone 0.9559,
combination 0.9682. The combination is worth 0.94 points on the test set. It is an
ensemble at almost zero parameter cost, which is what keeps it inside the budget:
two independent encoders would cost 9.2M parameters.

### What did not work, and why

The negative results are as informative as the positive ones, and three of them
redirected the work.

**Regularization is not the lever** (Step 5). Doubling the dropout and extending
training by 50% leaves the train/val gap at 5.5 points, unchanged. This is what
motivated pre-training: if more regularization does not close the gap, the missing
ingredient is supervision.

**Pre-training length saturates** (Step 7b). Between 12 and 16 epochs the gain is
0.04 points, that is noise. A 32-epoch run was therefore not attempted.

**Lambda is a slider, not a hyperparameter** (Step 7d). Raising it from 0.3 to 1.0
does improve the language model exactly as predicted, `val_mlm_loss` from 0.2051 to
0.1951 and PLL alone from 0.9559 to 0.9574, but the pointer loses just as much and
the combination does not move. The two heads share one encoder and the trade-off is
nearly zero-sum.

**Post-hoc calibration adds nothing** (Step 8c). Neither a per-position gamma nor a
temperature on the pointer logits beats the single gamma at unit temperature. The
limit lies in the two scores, not in how they are combined, which is what led to
testing lambda in the first place.

### Compliance with the constraints

- No external chess knowledge: SAN moves are atomic tokens, with no decomposition,
  no legality checks and no hand-built features. The model reaches 68.1% accuracy
  at reconstructing masked moves having learned the regularities from the data
  alone.
- No externally pre-trained model: pre-training is done from scratch on the 400,000
  provided games.
- `correct_move` is used only as a label to reconstruct the clean training games,
  never as a model input. The MLM head predicts randomly masked tokens in clean
  games, not `correct_move` as a designated target.
- 5,089,705 trainable parameters, under the 6,000,000 limit.

## STEP 12 - Exporting the weights

Utility cell, only needed when re-training: it lists the checkpoints present in
the working directory and, on Jupyter or Kaggle, produces download links.

`joint.weights.h5` is the only strictly necessary file: Step 10 rebuilds the whole
model from it, downloading it from the GitHub Release when it is not already on disk.
The other two are kept so that fine-tuning can be repeated without redoing the two
hours of pre-training.

If this cell reports the files as not present, the notebook is running the fast
path of Step 10 and no training checkpoint was produced in this session, which is
the expected behaviour.

In [ ]:
FILES = ["joint.weights.h5",              # final model, 5,089,705 parameters
         "encoder_mlm_e20.weights.h5",    # pre-trained encoder
         "mlm_full_e20.weights.h5"]       # encoder + MLM head

DOWNLOAD = False       # set to True to actually trigger the downloads

try:
    from google.colab import files as colab_files
    ON_COLAB = True
except ImportError:
    from IPython.display import FileLink, display
    ON_COLAB = False

for f in FILES:
    if not os.path.exists(f):
        print(f"not present: {f}  (produced by Steps 6 and 7c)")
        continue
    print(f"{os.path.getsize(f)/1e6:8.1f} MB  {f}")
    if not DOWNLOAD:
        continue
    if ON_COLAB:
        colab_files.download(f)           # browser download, Colab only
    else:
        display(FileLink(f))              # clickable link, Jupyter and Kaggle

    61.3 MB  joint.weights.h5
not present: encoder_mlm_e20.weights.h5  (produced by Steps 6 and 7c)
not present: mlm_full_e20.weights.h5  (produced by Steps 6 and 7c)
